# 24. The Static Truck Appointment System Problem

## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Goal
Implement Ant Colony Optimization (ACO) to provide a sophisticated metaheuristic approach that balances solution quality with computational efficiency. ACO uses swarm intelligence to explore the solution space and find high-quality assignments through collective learning.

### Key assumptions
- Artificial ants construct complete assignment solutions iteratively
- Pheromone trails guide subsequent ants toward promising assignments
- Balance between exploitation (pheromone) and exploration (heuristic) 
- Population-based search with multiple solutions in each iteration
- Adaptive learning through pheromone evaporation and reinforcement

### Approach (step-by-step)
1. **Initialize pheromone matrix** with uniform values
2. **Create ant colony** with specified number of artificial ants
3. **Construct solutions** using probabilistic selection based on pheromone and heuristic information
4. **Update pheromone trails** based on solution quality
5. **Apply evaporation** to forget poor solutions over time
6. **Track best solution** found during the search process
7. **Analyze convergence** behavior and parameter sensitivity

### What to look for in the results
- Convergence behavior showing improvement over iterations
- Pheromone trail patterns revealing good assignment choices
- Solution quality comparison with EDF heuristic and optimal solutions
- Parameter sensitivity (α, β, ρ) effects on performance
- Trade-offs between exploration and exploitation

### Concrete example (from the source)
The ACO algorithm demonstrates excellent performance through intelligent exploration of the solution space. Each ant constructs a complete assignment solution by iteratively selecting time slots for truck requests, with selection probability combining pheromone concentration and heuristic information.

**Expected Output:**
```
Iteration 0: Best=4.00, Avg=8.45
Iteration 20: Best=2.00, Avg=4.20
Iteration 40: Best=1.00, Avg=2.85
Iteration 60: Best=1.00, Avg=2.10

ACO Final Results:
Best Penalty: 1.00
Assignment:
Request 1: Slot 0 (preferred: 0, weight: 1)
Request 2: Slot 0 (preferred: 0, weight: 2)
Request 3: Slot 1 (preferred: 1, weight: 1)
```

In [ ]:
# Import required libraries for Ant Colony Optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for ACO implementation")

In [ ]:
# Define data structures (reuse from previous tiers)
@dataclass
class TruckRequest:
    """Represents a truck appointment request"""
    id: int
    preferred_time: int  # Preferred time slot (0-indexed)
    weight: float  # Priority weight
    service_type: str = 'standard'

@dataclass
class TimeSlot:
    """Represents a time slot with capacity constraints"""
    id: int
    capacity: int  # Maximum trucks that can be processed
    current_load: int = 0

@dataclass
class ACOSolution:
    """Represents a solution found by ACO"""
    assignments: Dict[int, int]  # request_id -> time_slot_id
    fitness: float  # Solution quality (lower is better)
    feasible: bool

print("Data structures defined for ACO metaheuristic")

In [ ]:
# Create extended test instances for ACO evaluation
def create_aco_test_instances():
    """Create test instances suitable for metaheuristic evaluation"""
    
    instances = {}
    
    # Instance 1: Extended concrete example (8 requests)
    requests1 = [
        TruckRequest(1, 0, 1.0), TruckRequest(2, 0, 2.0),
        TruckRequest(3, 1, 1.0), TruckRequest(4, 1, 1.0),
        TruckRequest(5, 2, 2.0), TruckRequest(6, 3, 1.0),
        TruckRequest(7, 2, 1.0), TruckRequest(8, 1, 2.0)
    ]
    time_slots1 = [TimeSlot(i, 2) for i in range(4)]
    instances['extended'] = (requests1, time_slots1)
    
    # Instance 2: Medium-sized problem
    requests2 = [TruckRequest(i, np.random.randint(0, 6), np.random.uniform(0.5, 2.0)) 
                 for i in range(1, 16)]
    time_slots2 = [TimeSlot(i, 3) for i in range(6)]
    instances['medium'] = (requests2, time_slots2)
    
    # Instance 3: Large problem for scalability test
    requests3 = [TruckRequest(i, np.random.randint(0, 8), np.random.uniform(0.5, 2.0)) 
                 for i in range(1, 26)]
    time_slots3 = [TimeSlot(i, 4) for i in range(8)]
    instances['large'] = (requests3, time_slots3)
    
    # Instance 4: Challenging capacity constraints
    requests4 = [
        TruckRequest(i, np.random.choice([0, 1, 2]), np.random.uniform(0.5, 2.0))
        for i in range(1, 13)
    ]
    time_slots4 = [TimeSlot(0, 3), TimeSlot(1, 3), TimeSlot(2, 2)]
    instances['tight'] = (requests4, time_slots4)
    
    return instances

aco_instances = create_aco_test_instances()
print(f"Created {len(aco_instances)} ACO test instances:")
for name, (reqs, slots) in aco_instances.items():
    print(f"  {name}: {len(reqs)} requests, {len(slots)} time slots")

In [ ]:
# Ant Colony Optimization Implementation
class AntColonyOptimization:
    """ACO for Static Truck Appointment System Problem"""
    
    def __init__(self, requests: List[TruckRequest], time_slots: List[TimeSlot],
                 num_ants: int = 20, max_iterations: int = 100,
                 alpha: float = 1.0, beta: float = 2.0, rho: float = 0.1,
                 q0: float = 0.9):  # Pseudorandom proportional rule parameter
        
        self.requests = requests
        self.time_slots = time_slots
        self.num_requests = len(requests)
        self.num_slots = len(time_slots)
        
        # ACO parameters
        self.num_ants = num_ants
        self.max_iterations = max_iterations
        self.alpha = alpha  # Pheromone influence
        self.beta = beta    # Heuristic information influence
        self.rho = rho      # Evaporation rate
        self.q0 = q0        # Pseudorandom threshold
        
        # Initialize pheromone matrix
        self.tau0 = 1.0 / (self.num_requests * self.num_slots)  # Initial pheromone
        self.pheromone = np.ones((self.num_requests, self.num_slots)) * self.tau0
        
        # Heuristic information matrix
        self.heuristic = self._calculate_heuristic()
        
        # Solution tracking
        self.best_solution = None
        self.best_fitness = float('inf')
        self.iteration_best = []
        self.iteration_avg = []
        self.convergence_history = []
        
        # Random seed for reproducibility
        np.random.seed(42)
        random.seed(42)
    
    def _calculate_heuristic(self) -> np.ndarray:
        """Calculate heuristic information matrix"""
        heuristic = np.zeros((self.num_requests, self.num_slots))
        
        for i, request in enumerate(self.requests):
            for j, slot in enumerate(self.time_slots):
                # Heuristic based on distance from preferred time
                distance = abs(j - request.preferred_time)
                heuristic[i, j] = 1.0 / (1.0 + distance)
        
        return heuristic
    
    def _calculate_fitness(self, assignments: Dict[int, int]) -> float:
        """Calculate fitness (penalty) for a given assignment"""
        if not self._is_feasible(assignments):
            return float('inf')
        
        total_penalty = 0.0
        
        # Deviation penalty
        for request in self.requests:
            assigned_slot = assignments[request.id]
            deviation = abs(assigned_slot - request.preferred_time)
            total_penalty += request.weight * deviation
        
        # Gate utilization balance penalty (simplified)
        slot_loads = {}
        for slot in self.time_slots:
            slot_loads[slot.id] = sum(1 for req in self.requests 
                                      if assignments[req.id] == slot.id)
        
        if slot_loads:
            max_load = max(slot_loads.values())
            min_load = min(slot_loads.values())
            total_penalty += 0.1 * (max_load - min_load)  # Small penalty for imbalance
        
        return total_penalty
    
    def _is_feasible(self, assignments: Dict[int, int]) -> bool:
        """Check if assignment respects capacity constraints"""
        for slot in self.time_slots:
            assigned_count = sum(1 for req in self.requests 
                               if assignments[req.id] == slot.id)
            if assigned_count > slot.capacity:
                return False
        return True
    
    def _construct_solution(self) -> ACOSolution:
        """Construct a complete solution using probabilistic rule"""
        assignments = {}
        remaining_capacity = {slot.id: slot.capacity for slot in self.time_slots}
        
        # Random order of request processing
        request_order = list(range(self.num_requests))
        np.random.shuffle(request_order)
        
        for req_idx in request_order:
            request = self.requests[req_idx]
            
            # Calculate probabilities for each time slot
            probabilities = []
            feasible_slots = []
            
            for slot_idx, slot in enumerate(self.time_slots):
                if remaining_capacity[slot.id] > 0:
                    # Calculate probability using pseudorandom proportional rule
                    pheromone = self.pheromone[req_idx, slot_idx]
                    heuristic = self.heuristic[req_idx, slot_idx]
                    probability = (pheromone ** self.alpha) * (heuristic ** self.beta)
                    
                    probabilities.append(probability)
                    feasible_slots.append(slot_idx)
            
            if not feasible_slots:
                # No feasible slots, assign randomly (will be infeasible)
                assigned_slot = np.random.randint(0, self.num_slots)
            else:
                # Normalize probabilities
                total_prob = sum(probabilities)
                if total_prob > 0:
                    probabilities = [p / total_prob for p in probabilities]
                    
                    # Pseudorandom proportional rule
                    if np.random.random() < self.q0:
                        # Exploitation: choose best feasible slot
                        best_idx = np.argmax(probabilities)
                        assigned_slot = feasible_slots[best_idx]
                    else:
                        # Exploration: probabilistic selection
                        chosen_idx = np.random.choice(len(feasible_slots), p=probabilities)
                        assigned_slot = feasible_slots[chosen_idx]
                else:
                    # All probabilities zero, choose randomly
                    assigned_slot = np.random.choice(feasible_slots)
            
            assignments[request.id] = assigned_slot
            remaining_capacity[assigned_slot] -= 1
        
        fitness = self._calculate_fitness(assignments)
        feasible = self._is_feasible(assignments)
        
        return ACOSolution(assignments, fitness, feasible)
    
    def _update_pheromone(self, solutions: List[ACOSolution]):
        """Update pheromone trails based on solution quality"""
        # Evaporation
        self.pheromone = (1 - self.rho) * self.pheromone
        
        # Deposit pheromone from all ants (elitist strategy with best solution)
        for solution in solutions:
            if solution.feasible:
                deposit_amount = 1.0 / solution.fitness
                
                for request in self.requests:
                    assigned_slot = solution.assignments[request.id]
                    req_idx = self.requests.index(request)
                    self.pheromone[req_idx, assigned_slot] += deposit_amount
        
        # Extra deposit for global best solution (elitism)
        if self.best_solution and self.best_solution.feasible:
            elite_deposit = 2.0 / self.best_solution.fitness  # Double deposit for best
            
            for request in self.requests:
                assigned_slot = self.best_solution.assignments[request.id]
                req_idx = self.requests.index(request)
                self.pheromone[req_idx, assigned_slot] += elite_deposit
    
    def optimize(self) -> ACOSolution:
        """Run the ACO optimization process"""
        print(f"Starting ACO optimization with {self.num_ants} ants, {self.max_iterations} iterations")
        print(f"Parameters: α={self.alpha}, β={self.beta}, ρ={self.rho}, q₀={self.q0}")
        
        for iteration in range(self.max_iterations):
            iteration_solutions = []
            iteration_fitness = []
            
            # Each ant constructs a solution
            for ant in range(self.num_ants):
                solution = self._construct_solution()
                iteration_solutions.append(solution)
                
                if solution.feasible:
                    iteration_fitness.append(solution.fitness)
                    
                    # Update global best solution
                    if solution.fitness < self.best_fitness:
                        self.best_solution = solution
                        self.best_fitness = solution.fitness
            
            # Update pheromone trails
            self._update_pheromone(iteration_solutions)
            
            # Track convergence
            if iteration_fitness:
                best_in_iter = min(iteration_fitness)
                avg_in_iter = np.mean(iteration_fitness)
                self.iteration_best.append(best_in_iter)
                self.iteration_avg.append(avg_in_iter)
                
                # Print progress every 10 iterations
                if (iteration + 1) % 10 == 0:
                    print(f"Iteration {iteration+1:3d}: Best={best_in_iter:.2f}, Avg={avg_in_iter:.2f}")
            
            self.convergence_history.append({
                'iteration': iteration + 1,
                'best': self.best_fitness if self.best_solution else float('inf'),
                'avg': avg_in_iter if iteration_fitness else float('inf')
            })
        
        return self.best_solution

print("ACO class defined successfully")

In [ ]:
# Run ACO on extended example
def run_aco_extended_example():
    """Run ACO on the extended example from source"""
    
    print("=" * 80)
    print("ANT COLONY OPTIMIZATION - EXTENDED EXAMPLE")
    print("=" * 80)
    
    requests, time_slots = aco_instances['extended']
    
    print("\nProblem Instance:")
    print(f"  Requests: {len(requests)}")
    print(f"  Time Slots: {len(time_slots)}")
    print(f"  Total Capacity: {sum(slot.capacity for slot in time_slots)}")
    
    print("\nRequest Details:")
    for req in requests:
        print(f"  Request {req.id}: Preferred slot {req.preferred_time}, Weight {req.weight}")
    
    print("\nTime Slot Capacities:")
    for slot in time_slots:
        print(f"  Slot {slot.id}: Capacity {slot.capacity}")
    
    # Run ACO optimization
    aco = AntColonyOptimization(
        requests, time_slots,
        num_ants=20,
        max_iterations=100,
        alpha=1.5,
        beta=2.0,
        rho=0.1,
        q0=0.9
    )
    
    start_time = time.time()
    best_solution = aco.optimize()
    execution_time = time.time() - start_time
    
    print("\n" + "="*60)
    print("ACO FINAL RESULTS")
    print("="*60)
    
    if best_solution:
        print(f"\nBest Penalty: {best_solution.fitness:.2f}")
        print(f"Execution Time: {execution_time:.2f} seconds")
        print(f"Solution Feasible: {best_solution.feasible}")
        
        print("\nAssignment Details:")
        for req in sorted(requests, key=lambda r: r.id):
            assigned_slot = best_solution.assignments[req.id]
            deviation = abs(assigned_slot - req.preferred_time)
            status = "✓" if deviation == 0 else f"(+{deviation})"
            print(f"  Request {req.id}: Slot {assigned_slot} (preferred: {req.preferred_time}, weight: {req.weight}) {status}")
        
        print("\nTime Slot Utilization:")
        for slot in time_slots:
            assigned_count = sum(1 for req in requests if best_solution.assignments[req.id] == slot.id)
            utilization = assigned_count / slot.capacity * 100
            print(f"  Slot {slot.id}: {assigned_count}/{slot.capacity} ({utilization:.1f}% utilized)")
        
        # Calculate statistics
        total_deviation = sum(abs(best_solution.assignments[req.id] - req.preferred_time) for req in requests)
        weighted_deviation = sum(req.weight * abs(best_solution.assignments[req.id] - req.preferred_time) for req in requests)
        
        print(f"\nSolution Statistics:")
        print(f"  Total Deviation: {total_deviation}")
        print(f"  Weighted Deviation: {weighted_deviation:.2f}")
        print(f"  Requests at preferred time: {sum(1 for req in requests if best_solution.assignments[req.id] == req.preferred_time)}/{len(requests)}")
        
        return aco, best_solution
    else:
        print("No feasible solution found")
        return aco, None

aco_solver, aco_best_solution = run_aco_extended_example()

In [ ]:
# Comprehensive visualization of ACO performance
def visualize_aco_performance(aco_solver, best_solution):
    """Create comprehensive visualizations of ACO performance"""
    
    if not aco_solver or not best_solution:
        print("No ACO solution available for visualization")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Ant Colony Optimization Performance Analysis', fontsize=16, fontweight='bold')
    
    requests, time_slots = aco_solver.requests, aco_solver.time_slots
    
    # 1. Convergence Plot
    ax1 = axes[0, 0]
    iterations = range(1, len(aco_solver.iteration_best) + 1)
    ax1.plot(iterations, aco_solver.iteration_best, 'b-', linewidth=2, label='Best')
    ax1.plot(iterations, aco_solver.iteration_avg, 'r--', linewidth=1, label='Average')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Fitness (Penalty)')
    ax1.set_title('ACO Convergence Behavior', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')  # Log scale for better visualization
    
    # 2. Pheromone Heatmap
    ax2 = axes[0, 1]
    pheromone_display = aco_solver.pheromone.T  # Transpose for better layout
    sns.heatmap(pheromone_display, annot=True, fmt='.3f', cmap='YlOrRd',
               xticklabels=[f'Req {r.id}' for r in requests],
               yticklabels=[f'Slot {s.id}' for s in time_slots],
               ax=ax2)
    ax2.set_title('Final Pheromone Trail Matrix\n(Higher = More Desirable)', fontweight='bold')
    ax2.set_xlabel('Truck Requests')
    ax2.set_ylabel('Time Slots')
    
    # 3. Assignment Matrix
    ax3 = axes[0, 2]
    assignment_matrix = [[0 for _ in time_slots] for _ in requests]
    for req in requests:
        assigned_slot = best_solution.assignments[req.id]
        assignment_matrix[req.id-1][assigned_slot] = 1
    
    sns.heatmap(assignment_matrix, annot=True, cmap='RdYlGn_r', cbar=False,
               xticklabels=[f'Slot {t.id}' for t in time_slots],
               yticklabels=[f'Req {r.id}' for r in requests],
               ax=ax3)
    ax3.set_title('Best Assignment Matrix\n(Green = Assigned)', fontweight='bold')
    ax3.set_xlabel('Time Slots')
    ax3.set_ylabel('Truck Requests')
    
    # 4. Heuristic vs Pheromone Comparison
    ax4 = axes[1, 0]
    # Select a sample request for detailed analysis
    sample_req = requests[0]
    req_idx = requests.index(sample_req)
    
    slots = list(range(len(time_slots)))
    pheromone_values = aco_solver.pheromone[req_idx, :]
    heuristic_values = aco_solver.heuristic[req_idx, :]
    
    x = np.arange(len(slots))
    width = 0.35
    
    ax4.bar(x - width/2, pheromone_values, width, label='Pheromone', alpha=0.7, color='orange')
    ax4.bar(x + width/2, heuristic_values, width, label='Heuristic', alpha=0.7, color='blue')
    
    ax4.set_xlabel('Time Slots')
    ax4.set_ylabel('Value')
    ax4.set_title(f'Pheromone vs Heuristic\n(Request {sample_req.id}, Preferred: {sample_req.preferred_time})', fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels([f'Slot {i}' for i in slots])
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Solution Quality Distribution
    ax5 = axes[1, 1]
    # Analyze last iteration solutions
    last_iteration_solutions = []
    for _ in range(100):  # Sample 100 solutions
        sol = aco_solver._construct_solution()
        if sol.feasible:
            last_iteration_solutions.append(sol.fitness)
    
    if last_iteration_solutions:
        ax5.hist(last_iteration_solutions, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
        ax5.axvline(best_solution.fitness, color='red', linestyle='--', linewidth=2, 
                   label=f'Best: {best_solution.fitness:.2f}')
        ax5.axvline(np.mean(last_iteration_solutions), color='green', linestyle='--', 
                   label=f'Mean: {np.mean(last_iteration_solutions):.2f}')
        ax5.set_xlabel('Fitness (Penalty)')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Solution Quality Distribution\n(Final Iteration)', fontweight='bold')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
    
    # 6. Parameter Sensitivity Summary
    ax6 = axes[1, 2]
    # Create a summary of ACO parameters
    params = ['α (Pheromone)', 'β (Heuristic)', 'ρ (Evaporation)', 'q₀ (Exploit)']
    values = [aco_solver.alpha, aco_solver.beta, aco_solver.rho, aco_solver.q0]
    colors = ['orange', 'blue', 'red', 'green']
    
    bars = ax6.bar(params, values, color=colors, alpha=0.7)
    ax6.set_ylabel('Parameter Value')
    ax6.set_title('ACO Parameter Settings', fontweight='bold')
    ax6.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed analysis
    print("\n" + "=" * 80)
    print("DETAILED ACO ANALYSIS")
    print("=" * 80)
    
    print("\n1. Convergence Analysis:")
    if len(aco_solver.iteration_best) > 10:
        initial_best = aco_solver.iteration_best[0]
        final_best = aco_solver.iteration_best[-1]
        improvement = (initial_best - final_best) / initial_best * 100
        print(f"   Initial best fitness: {initial_best:.2f}")
        print(f"   Final best fitness: {final_best:.2f}")
        print(f"   Improvement: {improvement:.1f}%")
        
        # Find convergence point (when improvement < 1% for 5 consecutive iterations)
        convergence_point = None
        for i in range(5, len(aco_solver.iteration_best)):
            recent_improvement = (aco_solver.iteration_best[i-5] - aco_solver.iteration_best[i]) / aco_solver.iteration_best[i-5]
            if recent_improvement < 0.01:  # Less than 1% improvement
                convergence_point = i
                break
        
        if convergence_point:
            print(f"   Convergence point: Iteration {convergence_point}")
        else:
            print(f"   Convergence: Not reached within {len(aco_solver.iteration_best)} iterations")
    
    print("\n2. Pheromone Pattern Analysis:")
    # Find strongest pheromone trails
    pheromone_flat = aco_solver.pheromone.flatten()
    top_indices = np.argsort(pheromone_flat)[-5:]  # Top 5 pheromone values
    
    print("   Top 5 pheromone trails:")
    for idx in reversed(top_indices):
        req_idx = idx // len(time_slots)
        slot_idx = idx % len(time_slots)
        req = requests[req_idx]
        slot = time_slots[slot_idx]
        pheromone_value = aco_solver.pheromone[req_idx, slot_idx]
        print(f"     Req {req.id} → Slot {slot.id}: {pheromone_value:.4f}")
    
    print("\n3. Algorithm Complexity:")
    print(f"   Time complexity: O(iterations × ants × requests × slots)")
    print(f"   Space complexity: O(requests × slots) for pheromone matrix")
    print(f"   Actual iterations: {len(aco_solver.iteration_best)}")
    print(f"   Population size: {aco_solver.num_ants} ants")

visualize_aco_performance(aco_solver, aco_best_solution)

In [ ]:
# Parameter sensitivity analysis
def parameter_sensitivity_analysis():
    """Analyze sensitivity of ACO parameters"""
    
    print("\n" + "=" * 80)
    print("ACO PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 80)
    
    requests, time_slots = aco_instances['extended']
    
    # Test different parameter combinations
    parameter_tests = [
        {'alpha': 1.0, 'beta': 1.0, 'rho': 0.1, 'q0': 0.9, 'name': 'Balanced'},
        {'alpha': 2.0, 'beta': 1.0, 'rho': 0.1, 'q0': 0.9, 'name': 'Pheromone-heavy'},
        {'alpha': 1.0, 'beta': 2.0, 'rho': 0.1, 'q0': 0.9, 'name': 'Heuristic-heavy'},
        {'alpha': 1.0, 'beta': 1.0, 'rho': 0.3, 'q0': 0.9, 'name': 'High evaporation'},
        {'alpha': 1.0, 'beta': 1.0, 'rho': 0.1, 'q0': 0.7, 'name': 'More exploration'},
        {'alpha': 1.5, 'beta': 2.0, 'rho': 0.1, 'q0': 0.9, 'name': 'Default'}
    ]
    
    results = {}
    
    print("\n1. Parameter Combination Testing:")
    print("   Configuration → Best Fitness, Convergence Iteration, Execution Time")
    
    for params in parameter_tests:
        # Run ACO with specific parameters
        aco_test = AntColonyOptimization(
            requests, time_slots,
            num_ants=15,  # Reduced for faster testing
            max_iterations=50,  # Reduced for faster testing
            alpha=params['alpha'],
            beta=params['beta'],
            rho=params['rho'],
            q0=params['q0']
        )
        
        start_time = time.time()
        solution = aco_test.optimize()
        execution_time = time.time() - start_time
        
        best_fitness = solution.fitness if solution else float('inf')
        convergence_iter = len(aco_test.iteration_best)
        
        results[params['name']] = {
            'fitness': best_fitness,
            'convergence': convergence_iter,
            'time': execution_time,
            'params': params
        }
        
        print(f"   {params['name']:>16} → {best_fitness:>8.2f}, {convergence_iter:>19}, {execution_time:>13.2f}s")
    
    # Find best configuration
    best_config = min(results.keys(), key=lambda k: results[k]['fitness'])
    print(f"\nBest configuration: {best_config}")
    print(f"Best fitness: {results[best_config]['fitness']:.2f}")
    
    # Parameter impact analysis
    print("\n2. Parameter Impact Analysis:")
    
    # Alpha sensitivity (pheromone influence)
    print("\n   Alpha (Pheromone Influence):")
    alpha_values = [0.5, 1.0, 1.5, 2.0, 3.0]
    for alpha in alpha_values:
        aco_alpha = AntColonyOptimization(
            requests, time_slots, num_ants=10, max_iterations=30,
            alpha=alpha, beta=2.0, rho=0.1, q0=0.9
        )
        solution = aco_alpha.optimize()
        fitness = solution.fitness if solution else float('inf')
        print(f"     α={alpha:>4} → Fitness: {fitness:>6.2f}")
    
    # Beta sensitivity (heuristic influence)
    print("\n   Beta (Heuristic Influence):")
    beta_values = [0.5, 1.0, 1.5, 2.0, 3.0]
    for beta in beta_values:
        aco_beta = AntColonyOptimization(
            requests, time_slots, num_ants=10, max_iterations=30,
            alpha=1.5, beta=beta, rho=0.1, q0=0.9
        )
        solution = aco_beta.optimize()
        fitness = solution.fitness if solution else float('inf')
        print(f"     β={beta:>4} → Fitness: {fitness:>6.2f}")
    
    # Rho sensitivity (evaporation rate)
    print("\n   Rho (Evaporation Rate):")
    rho_values = [0.05, 0.1, 0.2, 0.3, 0.5]
    for rho in rho_values:
        aco_rho = AntColonyOptimization(
            requests, time_slots, num_ants=10, max_iterations=30,
            alpha=1.5, beta=2.0, rho=rho, q0=0.9
        )
        solution = aco_rho.optimize()
        fitness = solution.fitness if solution else float('inf')
        print(f"     ρ={rho:>4} → Fitness: {fitness:>6.2f}")
    
    return results

sensitivity_results = parameter_sensitivity_analysis()

In [ ]:
# Comparison with other methods and scalability analysis
def comprehensive_comparison():
    """Compare ACO with other methods across different instances"""
    
    print("\n" + "=" * 80)
    print("COMPREHENSIVE METHOD COMPARISON")
    print("=" * 80)
    
    # Import EDF from Tier 2 for comparison
    class SimpleEDF:
        def __init__(self, requests, time_slots):
            self.requests = requests
            self.time_slots = time_slots
        
        def solve(self):
            assignments = {}
            sorted_requests = sorted(self.requests, key=lambda r: r.preferred_time)
            
            for slot in self.time_slots:
                slot.current_load = 0
            
            for request in sorted_requests:
                preferred_slot = self.time_slots[request.preferred_time]
                if preferred_slot.current_load < preferred_slot.capacity:
                    preferred_slot.current_load += 1
                    assignments[request.id] = preferred_slot.id
                else:
                    # Find nearest available
                    for offset in range(1, max(len(self.time_slots), request.preferred_time + 1)):
                        forward_idx = request.preferred_time + offset
                        if forward_idx < len(self.time_slots):
                            slot = self.time_slots[forward_idx]
                            if slot.current_load < slot.capacity:
                                slot.current_load += 1
                                assignments[request.id] = slot.id
                                break
                        backward_idx = request.preferred_time - offset
                        if backward_idx >= 0:
                            slot = self.time_slots[backward_idx]
                            if slot.current_load < slot.capacity:
                                slot.current_load += 1
                                assignments[request.id] = slot.id
                                break
            
            # Calculate fitness
            total_penalty = 0.0
            for request in self.requests:
                assigned_slot = assignments[request.id]
                deviation = abs(assigned_slot - request.preferred_time)
                total_penalty += request.weight * deviation
            
            return {'assignments': assignments, 'fitness': total_penalty}
    
    comparison_results = {}
    
    for instance_name, (requests, time_slots) in aco_instances.items():
        print(f"\n--- Instance: {instance_name.upper()} ---")
        print(f"Requests: {len(requests)}, Time Slots: {len(time_slots)}")
        
        instance_results = {}
        
        # EDF Heuristic
        print("\n  Running EDF heuristic...")
        start_time = time.time()
        edf = SimpleEDF(requests, time_slots)
        edf_solution = edf.solve()
        edf_time = time.time() - start_time
        
        instance_results['EDF'] = {
            'fitness': edf_solution['fitness'],
            'time': edf_time,
            'method': 'Heuristic'
        }
        
        print(f"     EDF Fitness: {edf_solution['fitness']:.2f}, Time: {edf_time:.4f}s")
        
        # ACO Metaheuristic
        print("  Running ACO metaheuristic...")
        start_time = time.time()
        aco_comp = AntColonyOptimization(
            requests, time_slots,
            num_ants=min(20, len(requests)),
            max_iterations=min(50, len(requests) * 2),
            alpha=1.5, beta=2.0, rho=0.1, q0=0.9
        )
        aco_solution = aco_comp.optimize()
        aco_time = time.time() - start_time
        
        instance_results['ACO'] = {
            'fitness': aco_solution.fitness if aco_solution else float('inf'),
            'time': aco_time,
            'method': 'Metaheuristic'
        }
        
        print(f"     ACO Fitness: {aco_solution.fitness if aco_solution else 'inf':.2f}, Time: {aco_time:.4f}s")
        
        # Calculate improvement
        if aco_solution and edf_solution['fitness'] > 0:
            improvement = (edf_solution['fitness'] - aco_solution.fitness) / edf_solution['fitness'] * 100
            print(f"     ACO improvement over EDF: {improvement:+.1f}%")
        
        comparison_results[instance_name] = instance_results
    
    # Summary table
    print("\n" + "="*60)
    print("SUMMARY COMPARISON TABLE")
    print("="*60)
    print("\nInstance          |    EDF    |    ACO    | Improvement")
    print("-" * 55)
    
    for instance_name in comparison_results:
        results = comparison_results[instance_name]
        edf_fit = results['EDF']['fitness']
        aco_fit = results['ACO']['fitness']
        
        if aco_fit < float('inf') and edf_fit > 0:
            improvement = (edf_fit - aco_fit) / edf_fit * 100
            improvement_str = f"{improvement:+6.1f}%"
        else:
            improvement_str = "  N/A  "
        
        print(f"{instance_name:>16} | {edf_fit:>8.2f} | {aco_fit:>8.2f} | {improvement_str}")
    
    # Scalability analysis
    print("\n" + "="*60)
    print("SCALABILITY ANALYSIS")
    print("="*60)
    print("\nProblem Size | EDF Time (ms) | ACO Time (ms) | ACO Speed Factor")
    print("-" * 60)
    
    sizes = [6, 12, 18, 24, 30]
    
    for size in sizes:
        # Generate test problem
        test_requests = [TruckRequest(i, np.random.randint(0, min(8, size//3)), 
                                      np.random.uniform(0.5, 2.0)) for i in range(1, size+1)]
        test_slots = [TimeSlot(j, max(2, size//6)) for j in range(min(8, size//3))]
        
        # Time EDF
        start = time.time()
        edf_test = SimpleEDF(test_requests, test_slots)
        edf_test.solve()
        edf_time_ms = (time.time() - start) * 1000
        
        # Time ACO (reduced iterations for scalability test)
        start = time.time()
        aco_test = AntColonyOptimization(
            test_requests, test_slots,
            num_ants=min(10, size),
            max_iterations=min(20, size),
            alpha=1.5, beta=2.0, rho=0.1, q0=0.9
        )
        aco_test.optimize()
        aco_time_ms = (time.time() - start) * 1000
        
        speed_factor = aco_time_ms / edf_time_ms if edf_time_ms > 0 else float('inf')
        
        print(f"{size:>11} | {edf_time_ms:>11.2f} | {aco_time_ms:>11.2f} | {speed_factor:>13.1f}x")
    
    return comparison_results

comparison_results = comprehensive_comparison()

### Why this Tier exists vs earlier Tiers
This Tier 3 provides advanced metaheuristic capabilities that address limitations of both mathematical optimization and simple heuristics:

- **Better solution quality**: Often outperforms simple heuristics like EDF
- **Scalability**: Handles larger instances than MIP while maintaining quality
- **Adaptive learning**: Pheromone trails capture problem-specific knowledge
- **Population-based search**: Explores multiple solution regions simultaneously
- **Balance of exploration/exploitation**: Avoids local optima through stochastic search

### Pros / Cons vs earlier Tiers
**Pros vs Tier 1 (MIP):**
- Much faster for large instances
- Can handle complex constraints that are difficult to formulate
- No solver dependency issues
- More flexible for problem modifications

**Pros vs Tier 2 (EDF):**
- Higher solution quality through search
- Adaptive learning from search history
- Better at escaping local optima
- Population-based diversity

**Cons:**
- No optimality guarantees
- Parameter tuning required
- Stochastic results (different runs may vary)
- More complex implementation
- Higher computational cost than simple heuristics

### When to use this Tier
- **Medium to large instances** where MIP is too slow but heuristics are inadequate
- **Quality-critical applications** requiring better solutions than simple heuristics
- **Complex problem variants** with difficult mathematical formulations
- **Dynamic environments** requiring adaptive search strategies
- **Research and development** exploring advanced optimization techniques
- **Production systems** where solution quality justifies computational cost